# KanKyouKen SDK — Getting Started

This notebook covers the full SDK surface:
- Connecting and querying events
- Filtering and pagination
- Exporting to DataFrame / CSV
- Posting events (`post_event`, `post_events`)
- Subscribing to live events (`subscribe_to_events`)

## Setup

Install dependencies from the repo root:
```bash
pip install -e "sdk[pandas]"
pip install -r examples/requirements.txt
```

Make sure your project-root `.env` has the Supabase keys (URL, anon key, service role key, JWT token).
The notebook creates its own study and seed data automatically via `notebook_setup.py`.

In [1]:
import pandas as pd
from datetime import datetime, timedelta, timezone
from notebook_setup import setup

client, STUDY_ID, PARTICIPANT_IDS = setup(n_participants=3, n_events=30)

URL:          http://127.0.0.1:54321
Study:        86fbb60f-f28b-41df-80e0-e87438b02568
Participants: 3
Events:       30


## 1. Query Events

In [2]:
# Basic query — first page of 10
response = client.query_events(study_id=STUDY_ID, limit=10)

print(f'Total events in study: {response.pagination.total}')
print(f'Returned this page:    {response.pagination.returned}')
print()
for event in response.events:
    print(f'  {event.ts}  {event.event_type:<30}  participant={event.participant_id}')

Total events in study: 30
Returned this page:    10

  2026-02-12 19:42:42.981154+00:00  logout                          participant=d3189c82-fd1c-4b49-b72a-be470d324cd9
  2026-02-12 18:59:42.981154+00:00  logout                          participant=c02f7ae9-e905-4e1f-9ba0-b50bd177567d
  2026-02-11 17:49:42.981154+00:00  form_submit                     participant=c02f7ae9-e905-4e1f-9ba0-b50bd177567d
  2026-02-11 07:11:42.981154+00:00  login                           participant=df5efa42-5717-4697-bc99-a4cae3a0fe65
  2026-02-10 08:06:42.981154+00:00  form_submit                     participant=df5efa42-5717-4697-bc99-a4cae3a0fe65
  2026-02-10 04:44:42.981154+00:00  form_submit                     participant=c02f7ae9-e905-4e1f-9ba0-b50bd177567d
  2026-02-10 01:17:42.981154+00:00  page_view                       participant=c02f7ae9-e905-4e1f-9ba0-b50bd177567d
  2026-02-09 23:28:42.981154+00:00  page_view                       participant=d3189c82-fd1c-4b49-b72a-be470d324cd9
  2026-02-0

In [3]:
# Filter by event type and date range
date_from = datetime.now(timezone.utc) - timedelta(days=30)

response = client.query_events(
    study_id=STUDY_ID,
    event_type='page_view',
    date_from=date_from,
    limit=50,
)

print(f'page_view events in last 30 days: {response.pagination.total}')

page_view events in last 30 days: 7


## 2. Load All Events into a DataFrame

`iter_events` handles pagination automatically and yields flat `Event` objects.
`Event.to_dict()` flattens payload JSON fields into `payload_<key>` columns.

In [4]:
import pandas as pd

df = pd.DataFrame([e.to_dict() for e in client.iter_events(study_id=STUDY_ID)])

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

Shape: (30, 13)
Columns: ['id', 'participant_id', 'study_id', 'event_type', 'ts', 'session_id', 'app_version', 'platform', 'item_id', 'task_id', 'created_at', 'payload_page', 'payload_duration_ms']


,id,participant_id,study_id,event_type,ts,session_id,app_version,platform,item_id,task_id,created_at,payload_page,payload_duration_ms
0,4694a78a-bbd8-4114-862a-a74968b8a62c,d3189c82-fd1c-4b49-b72a-be470d324cd9,86fbb60f-f28b-41df-80e0-e87438b02568,logout,2026-02-12 19:42:42.981154+00:00,None,None,None,None,None,2026-02-12 09:25:43.382737+00:00,NaN,NaN
1,1193ca40-2fcb-4ad1-984a-b4f5f5fc297d,c02f7ae9-e905-4e1f-9ba0-b50bd177567d,86fbb60f-f28b-41df-80e0-e87438b02568,logout,2026-02-12 18:59:42.981154+00:00,None,None,None,None,None,2026-02-12 09:25:43.074043+00:00,NaN,NaN
2,ca8472f6-ecc9-45d7-b2f7-e52485914a63,c02f7ae9-e905-4e1f-9ba0-b50bd177567d,86fbb60f-f28b-41df-80e0-e87438b02568,form_submit,2026-02-11 17:49:42.981154+00:00,None,None,None,None,None,2026-02-12 09:25:43.098775+00:00,NaN,NaN
3,1749e9fd-b0e1-43c8-ae83-adad6a347fca,df5efa42-5717-4697-bc99-a4cae3a0fe65,86fbb60f-f28b-41df-80e0-e87438b02568,login,2026-02-11 07:11:42.981154+00:00,None,None,None,None,None,2026-02-12 09:25:43.244192+00:00,NaN,NaN
4,d0d56676-ba3b-416d-909b-00b81c05657c,df5efa42-5717-4697-bc99-a4cae3a0fe65,86fbb60f-f28b-41df-80e0-e87438b02568,form_submit,2026-02-10 08:06:42.981154+00:00,None,None,None,None,None,2026-02-12 09:25:43.433595+00:00,NaN,NaN


In [5]:
# Export directly to CSV (skips the manual concat step)
client.query_events(study_id=STUDY_ID, limit=1000).to_csv('events_export.csv')
print('Saved events_export.csv')

Saved events_export.csv


## 3. Post Events

Use `post_event()` to record a single event, or `post_events()` for a batch.
This is the same endpoint the research app uses.

In [6]:
# Post a single event
result = client.post_event(
    study_id=STUDY_ID,
    participant_id=PARTICIPANT_IDS[0],
    event_type='notebook_test',
    payload={'source': 'notebook', 'value': 42},
)

print(f'Stored event ID: {result.event_id}')
print(f'Created at:      {result.created_at}')

Stored event ID: e7c1d7df-00e5-416a-897c-5786347df6b5
Created at:      2026-02-12 09:25:44.036797+00:00


In [7]:
# Batch posting
events = [
    {
        'study_id': STUDY_ID,
        'participant_id': PARTICIPANT_IDS[0],
        'event_type': 'batch_item',
        'payload': {'index': i, 'value': i * 10},
    }
    for i in range(5)
]

results = client.post_events(events)
print(f'Posted {len(results)} events')
for r in results:
    print(f'  {r.event_id}')

Posted 5 events
  6fe25af4-571d-4fee-9a3e-024fea796a53
  230ee24e-37cf-4abd-bce5-1887ee74e038
  87116800-705e-4ff5-8e3e-415feeb56440
  48549528-2f1f-45d2-92e8-64d55f7911f9
  96ca06f6-8001-4624-b6f0-a2e4dd73948f


## 4. Subscribe to Live Events

`subscribe_to_events()` polls at a configurable interval and yields new events as they arrive.
It runs forever — use it in a loop with a break condition, or in a background thread.

The example below collects the next 3 events then stops (useful for demos and testing).

In [8]:
print('Waiting for events (post some from another script or notebook to see them arrive)...')
print('Polling every 10s. Stops after 3 events.\n')

received = []

for event in client.subscribe_to_events(study_id=STUDY_ID, poll_interval=10):
    received.append(event)
    print(f'  [{len(received)}] {event.ts}  {event.event_type}  participant={event.participant_id}')
    if len(received) >= 3:
        break

print(f'\nReceived {len(received)} event(s).')

Waiting for events (post some from another script or notebook to see them arrive)...
Polling every 10s. Stops after 3 events.

  [1] 2026-02-12 19:42:42.981154+00:00  logout  participant=d3189c82-fd1c-4b49-b72a-be470d324cd9
  [2] 2026-02-12 18:59:42.981154+00:00  logout  participant=c02f7ae9-e905-4e1f-9ba0-b50bd177567d
  [3] 2026-02-12 19:42:42.981154+00:00  logout  participant=d3189c82-fd1c-4b49-b72a-be470d324cd9

Received 3 event(s).
